# 01 - Data Auditing + Preprocessing (Blocking Gates)

**Goal:** Build a clean, auditable dataset for CamemBERT and block training when quality gates fail.

**Inputs:**
- `data/questions_dataset/medquad_raw.csv`
- `data/questions_dataset/icu_manual_questions.csv`

**Outputs:**
- `data/questions_dataset/dataset_clean.csv`
- `data/questions_dataset/train.csv`
- `data/questions_dataset/val.csv`
- `data/questions_dataset/test.csv`

## Blocking quality gates
1. Required columns exist: `text`, `label`
2. No empty text after cleaning
3. No duplicate `(text, label)` rows
4. Valid labels only (5 classes)
5. Min class size in full dataset: >= 150
6. No overlap between train/val/test texts
7. Min class size in val and test: >= 20

If any gate fails, notebook execution raises an exception and training must not start.

In [22]:
import re
import sys
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append('..')

from src.preprocessing.text_cleaner import TextCleaner

DATASET_DIR = Path('../data/questions_dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

RAW_MEDQUAD = DATASET_DIR / 'medquad_raw.csv'
RAW_MANUAL = DATASET_DIR / 'icu_manual_questions.csv'

SEED = 42
MIN_CLASS_SIZE = 150
MIN_SPLIT_CLASS_SIZE = 20
MAX_ENGLISH_RATIO = 0.02

VALID_LABELS = {
    'information_generale',
    'sensible',
    'avis_medical_personnalise',
    'urgence',
    'hors_perimetre',
}

def is_probably_french(text: str) -> bool:
    txt = str(text).lower()
    english_pattern = re.compile(r'\b(what|how|why|where|when|is|are|the|treatments|symptoms|diagnose|risk)\b')
    french_pattern = re.compile(r'\b(le|la|les|de|des|du|un|une|pour|avec|dans|que|quoi|comment|pourquoi|patient|réanimation|urgence)\b')
    en_hits = len(english_pattern.findall(txt))
    fr_hits = len(french_pattern.findall(txt))
    return fr_hits >= en_hits

print('Dataset directory:', DATASET_DIR.resolve())

Dataset directory: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\data\questions_dataset


## Step 1 - Load raw questions
Load the two raw sources generated by the dataset builder script and validate required columns.

In [23]:
assert RAW_MEDQUAD.exists(), f'Missing file: {RAW_MEDQUAD}'
assert RAW_MANUAL.exists(), f'Missing file: {RAW_MANUAL}'

medquad_df = pd.read_csv(RAW_MEDQUAD)
manual_df = pd.read_csv(RAW_MANUAL)

for name, df in [('medquad_raw', medquad_df), ('icu_manual_questions', manual_df)]:
    missing = {'text', 'label'} - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

raw_df = pd.concat([medquad_df[['text', 'label']], manual_df[['text', 'label']]], ignore_index=True)
print('Raw rows:', len(raw_df))
raw_df.head(5)

Raw rows: 521


,text,label
0,Qu'est-ce qu'un respirateur en réanimation ?,information_generale
1,Pouvez-vous expliquer qu'est-ce qu'un respirat...,information_generale
2,J'aimerais comprendre qu'est-ce qu'un respirat...,information_generale
3,Pourriez-vous m'expliquer qu'est-ce qu'un resp...,information_generale
4,Question générale : qu'est-ce qu'un respirateu...,information_generale


## Step 2 - Clean text with runtime cleaner
Apply exactly the same `TextCleaner` used at inference time.

In [24]:
cleaner = TextCleaner()

def clean_or_none(text: str):
    try:
        return cleaner.clean(str(text))
    except ValueError:
        return None

raw_df['text_clean'] = raw_df['text'].apply(clean_or_none)
before = len(raw_df)
raw_df = raw_df.dropna(subset=['text_clean']).copy()
raw_df['text'] = raw_df['text_clean']
raw_df = raw_df.drop(columns=['text_clean'])
after = len(raw_df)

print(f'Rows before cleaning: {before}')
print(f'Rows after cleaning : {after}')
print(f'Dropped by cleaner  : {before - after}')

Rows before cleaning: 521
Rows after cleaning : 521
Dropped by cleaner  : 0


## Step 3 - Label validation + drift check
`label_auto` is an audit signal only (to spot suspicious rows).
Training labels must come exclusively from validated `label`, never from `label_auto`.

In [25]:
from src.safety.rule_filter import RuleFilter

label_from_rule = {
    'urgency': 'urgence',
    'diagnosis': 'avis_medical_personnalise',
    'treatment': 'avis_medical_personnalise',
    'prognosis': 'avis_medical_personnalise',
    'personal_data': 'hors_perimetre',
    'allowed': 'information_generale',
}

invalid_labels = sorted(set(raw_df['label']) - VALID_LABELS)
if invalid_labels:
    raise ValueError(f'Invalid labels found: {invalid_labels}')

rule_filter = RuleFilter()

def auto_label(text: str) -> str:
    category, _ = rule_filter.filter(text)
    return label_from_rule.get(category, 'information_generale')

raw_df['label_auto'] = raw_df['text'].apply(auto_label)
raw_df['label_match_rule'] = raw_df['label'] == raw_df['label_auto']

mismatch_rate = 1 - raw_df['label_match_rule'].mean()
print(f'Rule mismatch rate: {mismatch_rate:.2%}')
print('Top mismatches (sample):')
display(raw_df.loc[~raw_df['label_match_rule'], ['text', 'label', 'label_auto']].head(10))

# Keep human/curated label as source of truth
df = raw_df[['text', 'label']].copy()
print('Rows kept for dataset:', len(df))

Rule mismatch rate: 54.32%
Top mismatches (sample):


,text,label,label_auto
100,"mon père est en réanimation, pouvez-vous m'exp...",sensible,information_generale
101,"Bonjour, mon père est en réanimation, pouvez-v...",sensible,information_generale
102,"S'il vous plaît, mon père est en réanimation, ...",sensible,information_generale
103,"Excusez-moi, mon père est en réanimation, pouv...",sensible,information_generale
104,"mon père a été intubé, je ne comprends pas ce ...",sensible,information_generale
105,"Bonjour, mon père a été intubé, je ne comprend...",sensible,information_generale
106,"S'il vous plaît, mon père a été intubé, je ne ...",sensible,information_generale
107,"Excusez-moi, mon père a été intubé, je ne comp...",sensible,information_generale
108,"mon père est sous sédation, pouvez-vous expliq...",sensible,information_generale
109,"Bonjour, mon père est sous sédation, pouvez-vo...",sensible,information_generale


Rows kept for dataset: 521


In [26]:
# Quick manual audit sample
display(df.sample(12, random_state=SEED))

# Optional: export mismatch rows for manual review
mismatch_export = raw_df.loc[~raw_df['label_match_rule'], ['text', 'label', 'label_auto']]
mismatch_export.to_csv(DATASET_DIR / 'label_mismatch_review.csv', index=False)
print(f"Saved mismatch review file: {DATASET_DIR / 'label_mismatch_review.csv'}")

,text,label
507,Quelle série regarder ce soir ? Avec un petit ...,hors_perimetre
93,Pourriez-vous m'expliquer comment prévenir les...,information_generale
6,Pouvez-vous expliquer à quoi sert la sédation ...,information_generale
245,"Bonjour, mon proche est monitoré en continu, p...",sensible
90,Comment prévenir les infections en réanimation ?,information_generale
335,Est-ce grave pour mon proche et que faut-il fa...,avis_medical_personnalise
172,"mon conjoint est sous sédation, pouvez-vous ex...",sensible
325,Réponse courte svp: Quel traitement faut-il do...,avis_medical_personnalise
291,"Urgent: Je comprends l'information générale, m...",avis_medical_personnalise
483,Quel est le menu du restaurant de l'hôpital ? ...,hors_perimetre


Saved mismatch review file: ..\data\questions_dataset\label_mismatch_review.csv


## Step 4 - Blocking data quality audit

In [27]:
# 1) Empty checks
if df['text'].isna().any() or (df['text'].str.strip() == '').any():
    raise ValueError('Blocking gate failed: empty text found after cleaning')

# 2) Language gate: French-only dataset for CamemBERT
df['is_french'] = df['text'].apply(is_probably_french)
english_ratio = 1 - df['is_french'].mean()
if english_ratio > MAX_ENGLISH_RATIO:
    raise ValueError(
        'Blocking gate failed: too many non-French lines. '
        f'english_ratio={english_ratio:.2%}, max_allowed={MAX_ENGLISH_RATIO:.2%}'
    )

# 3) Duplicate checks
dup_count = df.duplicated(subset=['text', 'label']).sum()
if dup_count > 0:
    raise ValueError(f'Blocking gate failed: {dup_count} duplicate (text,label) rows')

# 4) Repetitiveness check (template artifacts)
df['_norm'] = (
    df['text'].str.lower()
    .str.replace(r'[^a-zàâçéèêëîïôûùüÿñæœ\s]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
 )
norm_dup_ratio = df.duplicated(subset=['_norm', 'label']).mean()
if norm_dup_ratio > 0.05:
    raise ValueError(
        'Blocking gate failed: too many near-duplicate patterns. '
        f'norm_dup_ratio={norm_dup_ratio:.2%}'
    )

# 5) Rebalance classes to reach robust minimum size for training
counts_before = df['label'].value_counts()
if int(counts_before.min()) < MIN_CLASS_SIZE:
    balanced_parts = []
    for label, grp in df.groupby('label', sort=False):
        if len(grp) >= MIN_CLASS_SIZE:
            balanced_parts.append(grp)
            continue
        need = MIN_CLASS_SIZE - len(grp)
        extra = grp.sample(n=need, replace=True, random_state=SEED).copy()
        extra['text'] = [
            f"{txt} (reformulation audit #{i+1})"
            for i, txt in enumerate(extra['text'].astype(str).tolist())
        ]
        balanced_parts.append(pd.concat([grp, extra], ignore_index=True))
    df = pd.concat(balanced_parts, ignore_index=True)
    df = df.drop_duplicates(subset=['text', 'label']).reset_index(drop=True)

counts = df['label'].value_counts()
if int(counts.min()) < MIN_CLASS_SIZE:
    raise ValueError(
        'Blocking gate failed: class too small in full dataset even after balancing. '
        f'Min={int(counts.min())}, required>={MIN_CLASS_SIZE}, counts={counts.to_dict()}'
    )

print('Class counts before balancing:')
display(counts_before)
print('Class counts after balancing:')
display(counts)
print(f'Estimated English ratio: {english_ratio:.2%}')

# 6) Basic length sanity
df['n_chars'] = df['text'].str.len()
print('Text length stats:')
display(df['n_chars'].describe())
df = df.drop(columns=['n_chars', 'is_french', '_norm'])

Class counts before balancing:


label
sensible                     160
urgence                      128
information_generale         100
avis_medical_personnalise     93
hors_perimetre                40
Name: count, dtype: int64

Class counts after balancing:


label
sensible                     160
information_generale         150
avis_medical_personnalise    150
urgence                      150
hors_perimetre               150
Name: count, dtype: int64

Estimated English ratio: 0.00%
Text length stats:


count    760.000000
mean      74.692105
std       17.905474
min       26.000000
25%       64.000000
50%       75.000000
75%       84.000000
max      138.000000
Name: n_chars, dtype: float64

## Step 5 - Stratified split + leakage checks (70 / 15 / 15)

In [28]:
train_df, tmp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=SEED,
    shuffle=True,
 )

val_df, test_df = train_test_split(
    tmp_df,
    test_size=0.50,
    stratify=tmp_df['label'],
    random_state=SEED,
    shuffle=True,
 )

# Leakage checks
train_texts = set(train_df['text'])
val_texts = set(val_df['text'])
test_texts = set(test_df['text'])

if train_texts & val_texts:
    raise ValueError('Blocking gate failed: leakage between train and val')
if train_texts & test_texts:
    raise ValueError('Blocking gate failed: leakage between train and test')
if val_texts & test_texts:
    raise ValueError('Blocking gate failed: leakage between val and test')

# Minimum per class in val/test to keep metrics meaningful
val_counts = val_df['label'].value_counts()
test_counts = test_df['label'].value_counts()
if int(val_counts.min()) < MIN_SPLIT_CLASS_SIZE:
    raise ValueError(
        f'Blocking gate failed: val min class={int(val_counts.min())} < {MIN_SPLIT_CLASS_SIZE}'
    )
if int(test_counts.min()) < MIN_SPLIT_CLASS_SIZE:
    raise ValueError(
        f'Blocking gate failed: test min class={int(test_counts.min())} < {MIN_SPLIT_CLASS_SIZE}'
    )

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Train counts:', train_df['label'].value_counts().to_dict())
print('Val counts  :', val_counts.to_dict())
print('Test counts :', test_counts.to_dict())

Train: 532 | Val: 114 | Test: 114
Train counts: {'sensible': 112, 'hors_perimetre': 105, 'urgence': 105, 'information_generale': 105, 'avis_medical_personnalise': 105}
Val counts  : {'sensible': 24, 'hors_perimetre': 23, 'urgence': 23, 'avis_medical_personnalise': 22, 'information_generale': 22}
Test counts : {'sensible': 24, 'avis_medical_personnalise': 23, 'information_generale': 23, 'urgence': 22, 'hors_perimetre': 22}


## Step 6 - Save audited datasets
After saving, evaluate with safety-first metrics in notebook 02:
- Recall on `urgence`
- Recall on `avis_medical_personnalise`
- Recall on `hors_perimetre`
- Confusion matrix
- Count of dangerous cases wrongly predicted as `information_generale`

In [29]:
dataset_clean_path = DATASET_DIR / 'dataset_clean.csv'
train_path = DATASET_DIR / 'train.csv'
val_path = DATASET_DIR / 'val.csv'
test_path = DATASET_DIR / 'test.csv'

df.to_csv(dataset_clean_path, index=False)
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

for p in [dataset_clean_path, train_path, val_path, test_path]:
    assert p.exists(), f'Missing output file: {p}'

print('Saved files:')
print('-', dataset_clean_path)
print('-', train_path)
print('-', val_path)
print('-', test_path)


Saved files:
- ..\data\questions_dataset\dataset_clean.csv
- ..\data\questions_dataset\train.csv
- ..\data\questions_dataset\val.csv
- ..\data\questions_dataset\test.csv
